In [0]:
passengers_day1 = [
(101,"Rahul Sharma","Hyderabad","Economy","India"),
(102,"Priya Reddy","Bangalore","Business","India"),
(103,"Amit Kumar","Mumbai","Economy","India"),
(104,"Sneha Patel","Delhi","Premium Economy","India"),
(105,"Farhan Ali","Chennai","Economy","India")
]

columns = [
"passenger_id",
"passenger_name",
"city",
"travel_class",
"country"
]

df_day1 = spark.createDataFrame(
    passengers_day1,
    columns
)

df_day1.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
passengers_day2 = [
(102,"Priya Reddy","Bangalore","First Class","India"),
(104,"Sneha Patel","Hyderabad","Premium Economy","India"),
(106,"Neha Singh","Pune","Economy","India"),
(107,"Arjun Verma","Kochi","Business","India")
]

df_day2 = spark.createDataFrame(
    passengers_day2,
    columns
)

df_day2.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
df_day1.write \
.mode("overwrite") \
.format("delta") \
.save("/Volumes/databrick_hexa_7405613493089439/default/delta-brick")

In [0]:
delta_df = spark.read.format("delta").load("/Volumes/databrick_hexa_7405613493089439/default/delta-brick")
print(delta_df.count())

5


In [0]:
delta_df.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(
    spark,
    "/Volumes/databrick_hexa_7405613493089439/default/delta-brick"
)

deltaTable.history().show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------+------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------+
|version|timestamp          |userId         |userName                                               |operation|operationParameters                                         |job |notebook          |queryHistoryStatementId             |clusterId               |readVersion|isolationLevel   |isBlindAppend|operationMetrics                                                                                                                       |userMetadata|engineInfo                                |
+-------+-

In [0]:
from delta.tables import DeltaTable

In [0]:
deltaTable = DeltaTable.forPath(
    spark,
    "/Volumes/databrick_hexa_7405613493089439/default/delta-brick"
)

In [0]:
deltaTable.alias("target").merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.read.format("delta") \
.load("/Volumes/databrick_hexa_7405613493089439/default/delta-brick") \
.filter("passenger_id = 102") \
.show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [0]:
spark.read.format("delta") \
.load("/Volumes/databrick_hexa_7405613493089439/default/delta-brick") \
.filter("passenger_id = 106") \
.show()

+------------+--------------+----+------------+-------+
|passenger_id|passenger_name|city|travel_class|country|
+------------+--------------+----+------------+-------+
|         106|    Neha Singh|Pune|     Economy|  India|
+------------+--------------+----+------------+-------+



In [0]:
latest_df = spark.read \
.format("delta") \
.load("/Volumes/databrick_hexa_7405613493089439/default/delta-brick")
latest_df.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
print(latest_df.count())

7


In [0]:
version0_df = spark.read \
.format("delta") \
.option("versionAsOf", 0) \
.load("/Volumes/databrick_hexa_7405613493089439/default/delta-brick")

version0_df.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
latest_df = spark.read \
.format("delta") \
.load("/Volumes/databrick_hexa_7405613493089439/default/delta-brick")

latest_df.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
print("Version 0 Count:")
print(version0_df.count())

Version 0 Count:
5


In [0]:
print("Latest Count:")
print(latest_df.count())

Latest Count:
7


In [0]:
version0_df.filter(
    "passenger_id = 102"
).show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore|    Business|  India|
+------------+--------------+---------+------------+-------+



In [0]:
latest_df.filter(
    "passenger_id = 102"
).show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [0]:
version0_df.filter(
    "passenger_id = 104"
).show()

+------------+--------------+-----+---------------+-------+
|passenger_id|passenger_name| city|   travel_class|country|
+------------+--------------+-----+---------------+-------+
|         104|   Sneha Patel|Delhi|Premium Economy|  India|
+------------+--------------+-----+---------------+-------+



In [0]:
deltaTable.history().show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
%sql
OPTIMIZE delta.`/Volumes/databrick_hexa_7405613493089439/default/delta-brick`

path,metrics
dbfs:/Volumes/databrick_hexa_7405613493089439/default/delta-brick,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781698849086, 1781698849581, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql
OPTIMIZE delta.`/Volumes/databrick_hexa_7405613493089439/default/delta-brick`
ZORDER BY (city)

path,metrics
dbfs:/Volumes/databrick_hexa_7405613493089439/default/delta-brick,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2027), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781698898094, 1781698898618, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(
    spark,
    "/Volumes/databrick_hexa_7405613493089439/default/delta-brick"
)

deltaTable.delete(
    "passenger_id = 105"
)

DataFrame[num_affected_rows: bigint]

In [0]:
deltaTable.history().show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
%sql
VACUUM delta.`/Volumes/databrick_hexa_7405613493089439/default/delta-brick`

path
dbfs:/Volumes/databrick_hexa_7405613493089439/default/delta-brick
